In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, RobustScaler
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score

# 1. Загрузка данных
train_path = '/kaggle/input/competitions/linear-regression-competition-2026/prices_train.csv'
test_path = '/kaggle/input/competitions/linear-regression-competition-2026/prices_test.csv'

train = pd.read_csv(train_path, index_col=0)
test = pd.read_csv(test_path, index_col=0)

y = train['Y house price of unit area']
X = train.drop('Y house price of unit area', axis=1)

# Заполняем пропуски значениями из train (без утечки данных)
X = X.fillna(X.median())
test = test.fillna(X.median())

# 2. Продвинутая генерация признаков (Feature Engineering)
def engineer_features(df):
    df = df.copy()
    
    # Географический центр района Синдиан / Новый Тайбэй
    center_lat, center_lon = 24.97, 121.535
    
    # Евклидово расстояние до центра города
    df['dist_to_center'] = np.sqrt((df['X5 latitude'] - center_lat)**2 + (df['X6 longitude'] - center_lon)**2)
    # Манхэттенское расстояние (по сетке улиц)
    df['manhattan_dist'] = np.abs(df['X5 latitude'] - center_lat) + np.abs(df['X6 longitude'] - center_lon)
    
    # Нелинейное влияние расстояния до метро
    df['log_mrt'] = np.log1p(df['X3 distance to the nearest MRT station'])
    df['inv_mrt'] = 1.0 / (df['X3 distance to the nearest MRT station'] + 10)
    
    # Взаимодействие метро и магазинов шаговой доступности
    df['stores_per_log_mrt'] = df['X4 number of convenience stores'] / (df['log_mrt'] + 1)
    df['stores_mult_inv_mrt'] = df['X4 number of convenience stores'] * df['inv_mrt']
    
    # Возраст здания
    df['house_age_sq'] = df['X2 house age'] ** 2
    df['is_new'] = (df['X2 house age'] < 5).astype(float)
    
    return df

X_feat = engineer_features(X)
test_feat = engineer_features(test)

# 3. Сетка значений альфа для авто-подбора
alphas = np.logspace(-2, 4, 200)

# 4. Пайплайн модели: полиномы 2-й степени -> робастное масштабирование -> RidgeCV
cv = KFold(n_splits=5, shuffle=True, random_state=42)

model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', RobustScaler()),
    ('ridge', RidgeCV(alphas=alphas, cv=cv, scoring='neg_mean_squared_error'))
])

# 5. Обучение
model.fit(X_feat, y)

best_alpha = model.named_steps['ridge'].alpha_
print(f"Оптимальный параметр регуляризации alpha: {best_alpha:.4f}")

# 6. Предсказание и защита от выхода за пределы
predictions = model.predict(test_feat)
predictions = np.clip(predictions, a_min=y.min(), a_max=None)

# 7. Формирование сабмита
submission = pd.DataFrame({
    'index': test.index,
    'Y house price of unit area': predictions
})

submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно создан!")